# Ames nonvoter review

144 records individually reviewed by Codex; 120 stratified + 24 targeted. This is not human-expert or full-paper review. Run from the TxAgent repository root with the vllm environment. The committed JSONL annotations are authored review decisions, never generated by the selection heuristics. Re-running the sampling cells replaces sample.jsonl; only do so under the pinned source/contract hashes.


In [ ]:
import hashlib, json
from pathlib import Path
root=Path('data/starling_data/ames/source_review_v3')
summary=json.loads((root/'summary.json').read_text())
assert hashlib.sha256(Path('tools/chembl_tool/tasks/ames/source_contract.py').read_bytes()).hexdigest()==summary['inputs']['source_contract_sha256']
for name, info in summary['inputs']['raw_files'].items():
    assert hashlib.sha256((root.parent/'raw_v1'/name).read_bytes()).hexdigest()==info['sha256']


## draw_sample.py
Selection only; this code does not label scientific eligibility.


In [ ]:
import collections
import hashlib
import json
from pathlib import Path

import pyarrow.parquet as pq
from tools.chembl_tool.tasks.ames import source_contract as c

ROOT = Path('data/starling_data/ames')
OUT = Path('data/starling_data/ames/source_review_v3')
reasons = [
    'special_context_requires_semantic_review',
    'effect_attribution_requires_relational_review',
    'activation_or_experiment_arms_require_review',
    'cited_or_previous_experiment_requires_provenance_review',
    'assay_method_comparison_requires_review',
    'inferred_subject_or_outcome_requires_review',
    'special_exposure_requires_condition_review',
    'prediction_or_mixed_experiment_requires_review',
    'unresolved_or_nonstandard_strain_panel',
    'unresolved_metabolic_activation',
    'source_requests_more_context',
    'unreviewed_qualifying_conditions',
]
quota = {r: 6 for r in reasons}
quota.update({r: 3 for r in ['nonprimary_or_reference_use', 'weak_positive_requires_outcome_review', 'nonbinary_source_result']})
quota['base_other_family_L2'] = 9
quota.update({f'{s}_L2':10 for s in ['ames_v1','ames_v2','ames_v3']})
audit = pq.read_table(ROOT/'canonical_v1/record_audit.parquet').to_pandas()
selected=[]
counts=collections.Counter()
for source in ['ames_base','ames_v1','ames_v2','ames_v3']:
    a=audit[audit.source_id.eq(source)].set_index('source_row_number')
    pools=collections.defaultdict(list)
    offset=0
    for batch in pq.ParquetFile(ROOT/'raw_v1'/f'{source}.parquet').iter_batches(batch_size=8192):
        for raw in batch.to_pylist():
            ordinal=offset
            offset+=1
            meta=a.loc[ordinal].to_dict()
            if meta['is_voter']:
                continue
            if source=='ames_base':
                reason=c.gold_candidate(source,raw).reason
                if raw.get('assay_family')!='bacterial_reverse_mutation':
                    stratum='base_other_family_L2' if meta['group_id']==c.NEAR else ''
                else:
                    stratum=reason
            else:
                reason='not_a_direct_gold_candidate'
                stratum=f'{source}_L2' if meta['group_id']==c.NEAR else ''
            if stratum not in quota:
                continue
            counts[stratum]+=1
            rid=f'{source}:{ordinal}'
            # Reproducible hash ordering; retain a bounded reservoir per stratum.
            rank=hashlib.sha256(('ames-vote-review-20260906:'+rid).encode()).hexdigest()
            item={'id':rid,'stratum':stratum,'gold_reason':reason,'current_group':meta['group_id'],'final_reason':meta['reason'],'identity_status':meta['identity_verification'],'parent_inchi_key':meta['parent_inchi_key'],'raw':raw,'rank':rank}
            pools[stratum].append(item)
            if len(pools[stratum])>800:
                pools[stratum]=sorted(pools[stratum],key=lambda x:x['rank'])[:400]
    for stratum, pool in sorted(pools.items()):
        pool.sort(key=lambda x:x['rank'])
        # Distinct PMIDs within each stratum; no preference for apparent acceptability.
        pmids=set()
        for item in pool:
            pmid=item['raw']['pmid']
            if pmid in pmids:
                continue
            pmids.add(pmid)
            selected.append(item)
            if len(pmids)==quota[stratum]:
                break
    print(source,offset,'sample so far',len(selected),flush=True)
for i,item in enumerate(selected,1):
    item['review_number']=i
(OUT/'sample.jsonl').write_text(''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in selected))
(OUT/'sampling.json').write_text(json.dumps({'seed':'ames-vote-review-20260906','sampling':'smallest SHA256 ranks; unique PMID within each stratum; bounded reservoirs; intentionally unequal strata, not a population error estimate','population_counts':dict(counts),'quotas':quota,'n_sample':len(selected)},indent=2))


## targeted_sample.py
Selection only; this code does not label scientific eligibility.


In [ ]:
import hashlib,json,re
from pathlib import Path
import pyarrow.parquet as pq
from tools.chembl_tool.tasks.ames import source_contract as c

root=Path('data/starling_data/ames')
out=Path('data/starling_data/ames/source_review_v3')
existing=[json.loads(x) for x in (out/'sample.jsonl').read_text().splitlines()]
used={x['id'] for x in existing}
audit=pq.read_table(root/'canonical_v1/record_audit.parquet').to_pandas()
all_pools={}
for source in ['ames_base','ames_v1','ames_v2','ames_v3']:
    meta=audit[audit.source_id.eq(source)&audit.group_id.eq(c.NEAR)]
    table=pq.read_table(root/'raw_v1'/f'{source}.parquet').take(meta.source_row_number.tolist())
    candidates=[]
    for raw,m in zip(table.to_pylist(),meta.to_dict('records')):
        if m['source_record_id'] in used:
            continue
        if source=='ames_base' and raw.get('assay_family')=='bacterial_reverse_mutation':
            continue
        body=c.text(raw.get('support_text'))
        # High-recall prioritization only: never produces a vote or semantic verdict.
        if not re.search(r'\bTA[ -]?\d|\bWP2|\bames\b',body,re.I):
            continue
        if not re.search(r'mutagen|revertant|reverse.mutation',body,re.I):
            continue
        score=3*bool(re.search(r'revertant',body,re.I))+2*bool(re.search(r'S[ -]?9|without.{0,25}activation|with.{0,25}activation',body,re.I))+bool(re.search(r'\bTA[ -]?\d|\bWP2',body,re.I))+bool(re.search(r'positive|negative|non.mutagen|mutagenic|dose',body,re.I))
        rank=hashlib.sha256(('targeted-20260906:'+m['source_record_id']).encode()).hexdigest()
        candidates.append({'id':m['source_record_id'],'stratum':source+'_targeted_direct','gold_reason':c.gold_candidate(source,raw).reason,'current_group':m['group_id'],'final_reason':m['reason'],'identity_status':m['identity_verification'],'parent_inchi_key':m['parent_inchi_key'],'raw':raw,'priority_score':score,'rank':rank})
    candidates.sort(key=lambda x:(-x['priority_score'],x['rank']))
    all_pools[source]=len(candidates)
    pmids=set()
    for x in candidates:
        if x['raw']['pmid'] in pmids:
            continue
        pmids.add(x['raw']['pmid'])
        x['review_number']=len(existing)+1
        existing.append(x)
        if len(pmids)==6:
            break
print(all_pools,'total',len(existing))
(out/'sample.jsonl').write_text(''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in existing))
(out/'targeted_sampling.json').write_text(json.dumps({'method':'purposefully selected high-priority records, six distinct PMIDs per source, score is retrieval prioritization not a scientific classification','pool_counts':all_pools,'script':'targeted_sample.py'},indent=2))


## Inspect authored reviews and reproduce counts
The sample is stratified disproportionately and then deliberately enriched. Do not extrapolate its disposition percentages to the source population.


In [ ]:
import pandas as pd
sample=pd.read_json(root/'sample.jsonl',lines=True)
reviews=pd.read_json(root/'audit_annotations.jsonl',lines=True)
assert len(sample)==len(reviews)==144
assert set(sample.id)==set(reviews.source_record_id)
display(reviews.disposition.value_counts())
display(reviews[['source_record_id','disposition','rationale']].head(10))
display(reviews[reviews.disposition.eq('candidate_current_condition')][['source_record_id','current_identity_status','proposed_outcome','remaining_requirements']])


In [ ]:
import sqlite3
translations={'already_represented_study_vote': '实验合格，但已有同研究同条件票', 'needs_context_or_primary_source': '需上下文或原始研究', 'identity_or_test_material_problem': '身份或受试物问题', 'candidate_richer_condition': '需补充条件定义的候选', 'not_a_usable_direct_vote': '本条不是可用的直接实验票', 'candidate_current_condition': '现有条件下的投票审核候选'}
conn=sqlite3.connect(':memory:')
conn.execute('CREATE TABLE review_decisions (source_record_id TEXT PRIMARY KEY, disposition TEXT NOT NULL)')
conn.executemany('INSERT INTO review_decisions VALUES (?,?)', [(x['source_record_id'], translations[x['disposition']]) for x in reviews.to_dict('records')])
sql='SELECT disposition, COUNT(*) AS records FROM main.review_decisions GROUP BY disposition ORDER BY records DESC, disposition'
display(pd.read_sql_query(sql,conn))


In [ ]:
sample_by_id=sample.set_index('id').to_dict('index')
detail=[]
for a in reviews.to_dict('records'):
    raw=sample_by_id[a['source_record_id']]['raw']
    detail.append({'number':a['review_number'],'id':a['source_record_id'],'pmid':str(a['source_pmid']),'decision':translations[a['disposition']],'rationale':a['rationale'],'identity':a['current_identity_status'],'support_text':raw['support_text'],'extra_details':raw.get('extra_details') or '', 'remaining_requirements':a.get('remaining_requirements') if isinstance(a.get('remaining_requirements'), str) else ''})
pd.DataFrame(detail).to_sql('review_records',conn,index=False,if_exists='replace')
sql='SELECT number, id, pmid, decision, rationale, identity, support_text, extra_details, remaining_requirements FROM main.review_records ORDER BY number'
display(pd.read_sql_query(sql,conn).head(10))
